In [3]:
import pandas as pd
import numpy as np
import joblib
import re
import os
import time
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

In [4]:
print("=" * 60)
print("  PELATIHAN MODEL NAÏVE BAYES - DETEKSI PESAN BERBAHAYA ")
print("=" * 60)

  PELATIHAN MODEL NAÏVE BAYES - DETEKSI PESAN BERBAHAYA 


In [5]:
print("\n[1] Memuat dataset...")
 
df = pd.read_csv('dataset_final.csv')
 
print(f"    Total data   : {len(df)} baris")
print(f"    Kolom        : {list(df.columns)}")
print(f"    Distribusi label:")
for label, count in df['label'].value_counts().items():
    print(f"      - {label}: {count} data")
print(f"    Distribusi kategori:")
for kat, count in df['kategori'].value_counts().items():
    print(f"      - {kat}: {count} data")


[1] Memuat dataset...
    Total data   : 1200 baris
    Kolom        : ['id', 'pesan', 'kategori', 'label']
    Distribusi label:
      - Berisiko: 800 data
      - Tidak Berisiko: 400 data
    Distribusi kategori:
      - Kata Kasar & Ancaman: 400 data
      - Transaksi Aman: 183 data
      - Phishing: 120 data
      - Jadwal & Kegiatan: 86 data
      - Social Engineering: 85 data
      - Penipuan: 78 data
      - Informasi Umum: 70 data
      - Pencurian Akun: 66 data
      - Percakapan Biasa: 61 data
      - Malware: 51 data


In [6]:
stemmer  = StemmerFactory().create_stemmer()
sw_base  = StopWordRemoverFactory().get_stop_words()
sw_extra = [
    'anda','kamu','saya','kami','kita','nya','ini','itu',
    'dengan','untuk','ada','akan','sudah','telah',
    'ya','yg','jg','gak','ga','deh','dong','nih','lah','sih',
    'ku','mu','klo','tapi','jadi','bisa','agar','juga',
]

In [7]:
KATA_KASAR = {
    # Alat kelamin & seksual
    'kontol','memek','pepek','titit','toket','ngentot','entot','ngewe',
    'colmek','coli','masturbasi','bokep','porno','telanjang','bugil',
    'binal','mesum','cabul','bejat',
    # Makian umum
    'anjing','bangsat','bajingan','brengsek','keparat','sialan',
    'babi','goblok','tolol','dungu','geblek','kampret',
    'asu','jancok','jancuk','cuk','kon','taik','tai','setan',
    'iblis','laknat','terkutuk','jahanam','lonte','sundal',
    'pelacur','jalang','murahan','perek',
    # Ancaman kekerasan verbal
    'kubunuh','mampusin','bacok','hajar','tonjok','siksa',
    'aniaya','habisi','musnahkan','gebuk','cekik',
    # Penghinaan SARA
    'kafir','rasis',
}
 
STOPWORDS = set(sw_base + sw_extra) - KATA_KASAR

In [8]:
POLA_URL = (
    r'(bit\.ly|s\.id|rb\.gy|t\.ly|cutt\.ly|tinyurl\.com'
    r'|shorturl\.at|bit\.do|ow\.ly|is\.gd|tiny\.cc'
    r'|[\w-]+\.xyz|[\w-]+\.site)\S*'
)
 
print(f"\n    Stopwords    : {len(STOPWORDS)} kata")
print(f"    Kata kasar   : {len(KATA_KASAR)} kata (blacklist)")


    Stopwords    : 138 kata
    Kata kasar   : 62 kata (blacklist)


In [9]:
print("\n[2] Case Folding...")
print("    Mengubah semua teks menjadi huruf kecil (lowercase)")
 
df['step_casefolding'] = df['pesan'].str.lower().str.strip()
 
print(f"\n    Contoh:")
print(f"    Sebelum : {df['pesan'].iloc[0]}")
print(f"    Sesudah : {df['step_casefolding'].iloc[0]}")


[2] Case Folding...
    Mengubah semua teks menjadi huruf kecil (lowercase)

    Contoh:
    Sebelum : dasar sialan
    Sesudah : dasar sialan


In [10]:
print("\n[3] Tokenisasi...")
print("    Menandai pola khusus lalu memecah teks menjadi token kata")
 
def tokenisasi(text):
    # Tandai URL mencurigakan
    text = re.sub(POLA_URL, 'URL_CURIGA', text)
    # Tandai nomor HP
    text = re.sub(r'\b0\d[\d\-]{8,12}\b', 'NOMOR_HP_ASING', text)
    # Tandai kode OTP (5-8 digit)
    text = re.sub(r'\b\d{5,8}\b', 'KODE_OTP', text)
    # Tandai nominal uang
    text = re.sub(r'rp[\s]?\d+[\.,]?\d*\s*(juta|ribu|rb)?', 'NOMINAL_UANG', text)
    # Hapus sisa angka dan karakter khusus
    text = re.sub(r'\b\d+\b', '', text)
    text = re.sub(r'[^a-z_\s]', ' ', text)
    return [t for t in text.split() if len(t) > 0]
 
df['step_tokenisasi'] = df['step_casefolding'].apply(tokenisasi)
 
print(f"\n    Contoh:")
print(f"    Sebelum : {df['step_casefolding'].iloc[0]}")
print(f"    Sesudah : {df['step_tokenisasi'].iloc[0]}")


[3] Tokenisasi...
    Menandai pola khusus lalu memecah teks menjadi token kata

    Contoh:
    Sebelum : dasar sialan
    Sesudah : ['dasar', 'sialan']


In [11]:
print("\n[4] Stopword Removal...")
print("    Membuang kata umum tidak bermakna")
print("    Catatan: kata kasar TIDAK dihapus agar tetap jadi fitur model")
 
def hapus_stopword(tokens):
    return [
        t for t in tokens
        if t.isupper()                          # token khusus (URL_CURIGA, dll)
        or t in KATA_KASAR                      # kata kasar dijaga
        or (t not in STOPWORDS and len(t) > 1)  # kata biasa
    ]
 
df['step_stopword'] = df['step_tokenisasi'].apply(hapus_stopword)
 
print(f"\n    Contoh:")
print(f"    Sebelum : {df['step_tokenisasi'].iloc[0]}")
print(f"    Sesudah : {df['step_stopword'].iloc[0]}")


[4] Stopword Removal...
    Membuang kata umum tidak bermakna
    Catatan: kata kasar TIDAK dihapus agar tetap jadi fitur model

    Contoh:
    Sebelum : ['dasar', 'sialan']
    Sesudah : ['dasar', 'sialan']


In [12]:
print("\n[5] Stemming...")
print("    Mengubah kata ke bentuk dasar (Algoritma Nazief-Adriani)")
print("    Token khusus & kata kasar SKIP stemming")
print("    Proses ini memerlukan waktu, harap tunggu...")
 
def stemming(tokens):
    return [
        t if (t.isupper() or t in KATA_KASAR)
        else stemmer.stem(t)
        for t in tokens
    ]
 
start = time.time()
df['step_stemming'] = df['step_stopword'].apply(stemming)
df['teks_bersih']   = df['step_stemming'].apply(
    lambda t: ' '.join(t) if t else 'PESAN_KOSONG'
)
elapsed = time.time() - start
print(f"    Selesai dalam {elapsed:.1f} detik")
 
# Tampilkan ringkasan per tahap untuk contoh Berisiko & Tidak Berisiko
print("\n    ── Ringkasan 5 Tahap Preprocessing ──")
for lbl in ['Berisiko', 'Tidak Berisiko']:
    i = df[df['label'] == lbl].index[0]
    print(f"\n    [{lbl}]")
    print(f"    Asli       : {df['pesan'][i]}")
    print(f"    CaseFold   : {df['step_casefolding'][i]}")
    print(f"    Tokenisasi : {df['step_tokenisasi'][i]}")
    print(f"    Stopword   : {df['step_stopword'][i]}")
    print(f"    Stemming   : {df['step_stemming'][i]}")
    print(f"    Teks Final : {df['teks_bersih'][i]}")


[5] Stemming...
    Mengubah kata ke bentuk dasar (Algoritma Nazief-Adriani)
    Token khusus & kata kasar SKIP stemming
    Proses ini memerlukan waktu, harap tunggu...
    Selesai dalam 23.7 detik

    ── Ringkasan 5 Tahap Preprocessing ──

    [Berisiko]
    Asli       : dasar sialan
    CaseFold   : dasar sialan
    Tokenisasi : ['dasar', 'sialan']
    Stopword   : ['dasar', 'sialan']
    Stemming   : ['dasar', 'sialan']
    Teks Final : dasar sialan

    [Tidak Berisiko]
    Asli       : Transfer berhasil. Rp45.000 telah dikirim ke rekening BRI atas nama Rina.
    CaseFold   : transfer berhasil. rp45.000 telah dikirim ke rekening bri atas nama rina.
    Tokenisasi : ['transfer', 'berhasil', '_', 'telah', 'dikirim', 'ke', 'rekening', 'bri', 'atas', 'nama', 'rina']
    Stopword   : ['transfer', 'berhasil', 'dikirim', 'rekening', 'bri', 'atas', 'nama', 'rina']
    Stemming   : ['transfer', 'hasil', 'kirim', 'rekening', 'bri', 'atas', 'nama', 'rina']
    Teks Final : transfer hasil k

In [13]:
print("\n[6] Split Data Training dan Testing (80:20)...")


[6] Split Data Training dan Testing (80:20)...


In [14]:
label_map = {'Berisiko': 1, 'Tidak Berisiko': 0}
X = df['teks_bersih']
y = df['label'].map(label_map)
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
 
print(f"    Data Training : {len(X_train)} data (80%)")
print(f"    Data Testing  : {len(X_test)} data (20%)")
print(f"    Label encoding: Berisiko=1, Tidak Berisiko=0")

    Data Training : 960 data (80%)
    Data Testing  : 240 data (20%)
    Label encoding: Berisiko=1, Tidak Berisiko=0


In [15]:
print("\n[7] Ekstraksi Fitur — TF-IDF...")
print("    Mengubah teks menjadi representasi angka (vektor bobot)")
 
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),   # unigram + bigram
    max_features=5000,    # 5000 fitur terpenting
    sublinear_tf=True,    # log TF
    min_df=1              # masukkan semua kata
)
 
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)
 
feature_names = tfidf.get_feature_names_out()
mean_tfidf    = X_train_tfidf.mean(axis=0).A1
top10_idx     = mean_tfidf.argsort()[::-1][:10]
 
print(f"\n    Dimensi matriks training  : {X_train_tfidf.shape}")
print(f"    Dimensi matriks testing   : {X_test_tfidf.shape}")
print(f"    Jumlah fitur (vocabulary) : {len(tfidf.vocabulary_)}")
print(f"    Top 10 fitur TF-IDF       : {[feature_names[i] for i in top10_idx]}")


[7] Ekstraksi Fitur — TF-IDF...
    Mengubah teks menjadi representasi angka (vektor bobot)

    Dimensi matriks training  : (960, 3497)
    Dimensi matriks testing   : (240, 3497)
    Jumlah fitur (vocabulary) : 3497
    Top 10 fitur TF-IDF       : ['lo', 'banget', 'hasil', 'akun', 'bulan', 'gue', 'dasar', 'pukul', 'bayar', 'masuk']


In [16]:
print("\n[8] Training Model Multinomial Naïve Bayes...")
print("    alpha = 0.3 (Laplace smoothing)")
 
nb_model = MultinomialNB(alpha=0.3)
nb_model.fit(X_train_tfidf, y_train)
print("    Model selesai dilatih")


[8] Training Model Multinomial Naïve Bayes...
    alpha = 0.3 (Laplace smoothing)
    Model selesai dilatih


In [17]:
print("\n[9] Evaluasi Model...")
 
y_pred = nb_model.predict(X_test_tfidf)
 
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test, y_pred, zero_division=0)
f1   = f1_score(y_test, y_pred, zero_division=0)
cm   = confusion_matrix(y_test, y_pred)
 
TP = cm[1][1]
FN = cm[1][0]
FP = cm[0][1]
TN = cm[0][0]


[9] Evaluasi Model...


In [18]:
print(f"\n    Accuracy  : {acc:.4f}  ({acc*100:.2f}%)")
print(f"    Precision : {prec:.4f}  ({prec*100:.2f}%)")
print(f"    Recall    : {rec:.4f}  ({rec*100:.2f}%)")
print(f"    F1-Score  : {f1:.4f}  ({f1*100:.2f}%)")
print()
print("    Confusion Matrix:")
print(f"    {'':26s}  Pred: Berisiko   Pred: Tidak Berisiko")
print(f"    {'Aktual: Berisiko':26s}  TP={TP:8d}       FN={FN:8d}")
print(f"    {'Aktual: Tidak Berisiko':26s}  FP={FP:8d}       TN={TN:8d}")
print()
print("    Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Tidak Berisiko', 'Berisiko']))


    Accuracy  : 0.9708  (97.08%)
    Precision : 0.9693  (96.93%)
    Recall    : 0.9875  (98.75%)
    F1-Score  : 0.9783  (97.83%)

    Confusion Matrix:
                                Pred: Berisiko   Pred: Tidak Berisiko
    Aktual: Berisiko            TP=     158       FN=       2
    Aktual: Tidak Berisiko      FP=       5       TN=      75

    Classification Report:
                precision    recall  f1-score   support

Tidak Berisiko       0.97      0.94      0.96        80
      Berisiko       0.97      0.99      0.98       160

      accuracy                           0.97       240
     macro avg       0.97      0.96      0.97       240
  weighted avg       0.97      0.97      0.97       240



In [19]:
print("[10] Menyimpan model ke file joblib...")
 
os.makedirs('model', exist_ok=True)
output_path = 'model/model_naive_bayes.joblib'
 
# PENTING: Simpan TANPA fungsi/lambda agar aman di-load di manapun
joblib.dump({
    'model':        nb_model,       # MultinomialNB
    'vectorizer':   tfidf,          # TfidfVectorizer
    'stopwords':    list(STOPWORDS),
    'kata_kasar':   list(KATA_KASAR),
    'pola_url':     POLA_URL,
    'label_map':    label_map,      # {'Berisiko':1, 'Tidak Berisiko':0}
    'metadata': {
        'total_data':   len(df),
        'train_size':   len(X_train),
        'test_size':    len(X_test),
        'accuracy':     round(acc,  4),
        'precision':    round(prec, 4),
        'recall':       round(rec,  4),
        'f1_score':     round(f1,   4),
        'confusion_matrix': cm.tolist(),
        'label_encoding':   '1=Berisiko, 0=Tidak Berisiko',
        'kategori':         list(df['kategori'].unique()),
        'preprocessing_steps': [
            '1. Case Folding (lowercase)',
            '2. Tokenisasi + penandaan URL_CURIGA/NOMOR_HP_ASING/KODE_OTP/NOMINAL_UANG',
            '3. Stopword Removal (Sastrawi + custom, kata kasar dijaga)',
            '4. Stemming Nazief-Adriani via Sastrawi (token khusus & kata kasar skip)',
            '5. Ekstraksi Fitur TF-IDF (1,2)-gram max 5000 fitur',
        ],
        'catatan': (
            'Sistem menggunakan 2 lapis deteksi: '
            '(1) Blacklist kata kasar → langsung Berisiko, '
            '(2) Model Naïve Bayes untuk pola ancaman lainnya. '
            'Pesan < 4 karakter langsung Tidak Berisiko (percakapan biasa).'
        )
    }
}, output_path)
 
size_kb = os.path.getsize(output_path) / 1024
print(f"    File    : {output_path}")
print(f"    Ukuran  : {size_kb:.1f} KB")

[10] Menyimpan model ke file joblib...
    File    : model/model_naive_bayes.joblib
    Ukuran  : 263.6 KB


In [21]:
print("\n[11] Uji Prediksi dengan data baru...")
 
# Fungsi prediksi final (sama persis dengan classification_service.py)
def prediksi(pesan):
    pesan = str(pesan).strip()
 
    # Rule 1: terlalu pendek
    if len(pesan) < 4:
        return {'label': 'Tidak Berisiko', 'keyakinan': 99.0, 'alasan': 'Terlalu pendek'}
 
    # Rule 2: blacklist kata kasar
    t      = re.sub(r'[^a-z\s]', ' ', pesan.lower())
    tokens = set(t.split())
    if tokens & KATA_KASAR:
        return {'label': 'Berisiko', 'keyakinan': 99.0, 'alasan': 'Kata kasar terdeteksi'}
 
    # Preprocessing
    t = pesan.lower().strip()
    t = re.sub(POLA_URL, 'URL_CURIGA', t)
    t = re.sub(r'\b0\d[\d\-]{8,12}\b', 'NOMOR_HP_ASING', t)
    t = re.sub(r'\b\d{5,8}\b', 'KODE_OTP', t)
    t = re.sub(r'rp[\s]?\d+[\.,]?\d*\s*(juta|ribu|rb)?', 'NOMINAL_UANG', t)
    t = re.sub(r'\b\d+\b', '', t)
    t = re.sub(r'[^a-z_\s]', ' ', t)
    tks = [x for x in t.split() if len(x) > 0]
    tks = [x for x in tks if x.isupper() or x in KATA_KASAR or (x not in STOPWORDS and len(x) > 1)]
    tks = [x if (x.isupper() or x in KATA_KASAR) else stemmer.stem(x) for x in tks]
    bersih = ' '.join(tks) if tks else 'PESAN_KOSONG'
 
    # Rule 3: token hampir kosong
    if len(tks) < 2:
        return {'label': 'Tidak Berisiko', 'keyakinan': 90.0, 'alasan': 'Token kosong'}
 
    # Model NB
    vec   = tfidf.transform([bersih])
    pred  = nb_model.predict(vec)[0]
    prob  = nb_model.predict_proba(vec)[0]
    label = 'Berisiko' if pred == 1 else 'Tidak Berisiko'
    return {'label': label, 'keyakinan': round(max(prob)*100, 2), 'alasan': 'Model NB'}
 
 
test_cases = [
    # Percakapan sehari-hari → Tidak Berisiko
    ("Tidak Berisiko", "Dimas Kurniawan"),
    ("Tidak Berisiko", "itu kan ada beberapa"),
    ("Tidak Berisiko", "ngentot Yuk"),
    ("Tidak Berisiko", "iya nanti gue hubungi balik ya"),
    ("Tidak Berisiko", "udah makan belum? tadi beli nasi padang enak"),
    ("Tidak Berisiko", "besok jadi berangkat kan? tunggu di stasiun jam 7"),
    ("Tidak Berisiko", "Transfer berhasil Rp250.000 ke rekening BCA atas nama Budi"),
    # Kata kasar → Berisiko
    ("Berisiko", "kontol lo!"),
    ("Berisiko", "dasar anjing"),
    ("Berisiko", "KONTOL!!"),
    ("Berisiko", "bangsat banget sih"),
    ("Berisiko", "goblok lo itu"),
    ("Berisiko", "jancok mau apa lo"),
    # Phishing & ancaman → Berisiko
    ("Berisiko", "Akun BRI terdeteksi mencurigakan. Verifikasi di bit.ly/verif"),
    ("Berisiko", "Selamat menang Rp50 juta. Klik link ini untuk klaim"),
    ("Berisiko", "Berikan kode OTP 819234 kepada saya untuk verifikasi"),
    ("Berisiko", "Download APK WhatsApp terbaru di shorturl.at/wa-update"),
    ("Berisiko", "Bisnis online profit 30% dijamin WA 081234567890"),
]
 
print(f"\n    {'Ekspektasi':16s}  {'Hasil':16s}  {'Yakin':7s}  Sts  Alasan / Pesan")
print("    " + "-" * 85)
 
benar = 0
for ekspektasi, pesan in test_cases:
    hasil  = prediksi(pesan)
    status = "✓ OK" if hasil['label'] == ekspektasi else "✗ SALAH"
    if hasil['label'] == ekspektasi:
        benar += 1
    print(f"    {ekspektasi:16s}  {hasil['label']:16s}  {hasil['keyakinan']:5.1f}%  "
          f"{status}  [{hasil['alasan']}] {pesan[:35]}")
 
print(f"\n    Hasil : {benar}/{len(test_cases)} benar ({benar/len(test_cases)*100:.0f}%)")
print("\n" + "=" * 60)
print("  SELESAI — model/model_naive_bayes.joblib siap digunakan")
print("=" * 60)


[11] Uji Prediksi dengan data baru...

    Ekspektasi        Hasil             Yakin    Sts  Alasan / Pesan
    -------------------------------------------------------------------------------------
    Tidak Berisiko    Berisiko           59.0%  ✗ SALAH  [Model NB] Dimas Kurniawan
    Tidak Berisiko    Berisiko           55.0%  ✗ SALAH  [Model NB] itu kan ada beberapa
    Tidak Berisiko    Berisiko           99.0%  ✗ SALAH  [Kata kasar terdeteksi] ngentot Yuk
    Tidak Berisiko    Tidak Berisiko     50.8%  ✓ OK  [Model NB] iya nanti gue hubungi balik ya
    Tidak Berisiko    Tidak Berisiko     97.7%  ✓ OK  [Model NB] udah makan belum? tadi beli nasi pa
    Tidak Berisiko    Tidak Berisiko     89.5%  ✓ OK  [Model NB] besok jadi berangkat kan? tunggu di
    Tidak Berisiko    Tidak Berisiko     98.1%  ✓ OK  [Model NB] Transfer berhasil Rp250.000 ke reke
    Berisiko          Berisiko           99.0%  ✓ OK  [Kata kasar terdeteksi] kontol lo!
    Berisiko          Berisiko           99.0% 